In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [60]:
pd.set_option('display.float_format', '{:.0f}'.format)

# 데이터 로드

In [61]:
#--국토교통부 실거래가공개시스템 상업/업무용 부동산 매매거래 데이터 수집

In [62]:
# (SQL에서 1차 전처리) -- transactions_total.csv
#--시군구, 법정동 분리
#--계약연월: 숫자 → 날짜타입으로 변환
#--거래금액: 문자 → 숫자타입으로 변환
#--지분거래 및 계약취소(`해제사유발생일`)건 제외(where `지분구분` is null and `해제사유발생일` is null)
#--컬럼 '지번', '`도로명` 제외(분석 단위: 법정동, 지번의 경우 마스킹된 정보 다수)
#--컬럼 `계약일` 제외(분석 단위: 월, 계약연월로 대체 가능)
#--컬럼 `도로조건`, `매수`, `매도`, `건축년도`, `거래유형`(직거래/중개거래 여부), `중개사소재지` (거래량 추출 위해 수집한 데이터셋이므로 분석에 불필요한 정보라 판단)
#--컬럼 `대지면적(㎡)` 제외(결측치 비율 92.29%)

In [63]:
file_path = 'C:/python_course/final_project/eunbi'
trans_df = pd.read_csv(file_path+'/transactions_total.csv', encoding = 'cp949')

In [64]:
trans_df.head()

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월
0,경기도 성남시 분당구,구미동,집합,근린상업,30,제2종근린생활,69500,1,2023-01
1,경기도 성남시 분당구,금곡동,집합,중심상업,45,제2종근린생활,69000,1,2023-01
2,경기도 성남시 분당구,서현동,집합,근린상업,20,교육연구,9753,2,2023-01
3,경기도 성남시 분당구,서현동,집합,근린상업,76,제1종근린생활,37000,NaN,2023-01
4,경기도 성남시 분당구,서현동,집합,근린상업,76,제1종근린생활,37000,2,2023-01


In [65]:
trans_df['유형'].value_counts()

유형
집합    2871
일반     189
Name: count, dtype: int64

- 집합 건물: 한 동의 건물에서 구조상 구분된 몇 개의 부분이 독립된 건물로 사용될 수 있어 구분 소유권의 객체가 될 수 있는 건물.

In [66]:
trans_df['용도지역'].value_counts()

용도지역
중심상업       679
준주거        587
일반상업       584
근린상업       484
제3종일반주거    332
제2종일반주거    207
기타          55
보전녹지        46
일반공업        33
자연녹지        27
제1종일반주거     13
용도미지정        8
제2종전용주거      2
일반주거         2
개발제한구역       1
Name: count, dtype: int64

- 용도지역: 국가가 땅의 쓰임새를 정해놓은 가이드라인(토지의 이용 목적과 건축물의 용도·건폐율·용적률·높이 등을 제한해 토지를 경제적·효율적으로 이용하고 공공복리를 증진하기 위해 도시관리계획으로 지정하는 지역)
<br>
- 왜 필요한가? 어떤 지역이 상업화되고 있는가 확인 가능. 젠트리케이션은 주로 주거지역이나 준공업지역이 상업적 성격으로 변모하며 발생, 만약 거래량이 폭증한 곳이 원래부터 '중심상업지역'이었다면 일반적인 상권 활성화일 가능성이 높으나 '제2종 일반주거지역' 내에서 상업용 부동산 거래량이 늘어난다면 이는 단독주택이 카페나 식당으로 개조되는 전형적인 젠트리케이션의 초기 신호로 볼 수 있음.

In [67]:
trans_df['건축물주용도'].value_counts()

건축물주용도
제2종근린생활    1075
제1종근린생활     883
업무          456
판매          305
기타          204
교육연구        126
숙박           11
Name: count, dtype: int64

- 제1종근린생활: 일용품을 판매하는 소매점, 이용원, 의원, 탁구장, 마을회관 등 주택가와 인접해 주민들의 생활 편의를 도울 수 있는 시설
- 제2종근린생활: 공연장, 종교집회장, 청소년게임제공업소, 금융업소, 사진관 등 주택가와 인접해 주민들의 생활 편의를 도울 수 있는 시설(1종근린생활보다 큰 규모 시설)

# 중복행 개수: 0개 

In [68]:
trans_df[trans_df.duplicated(subset=['시군구', '법정동', '유형', '용도지역', '전용/연면적(㎡)', '건축물주용도', '거래금액(만원)', '층', '계약연월'], keep=False)]

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월


중복행 없음(MySQL 활용 1차 전처리 과정에서 DISTINCT)

# 결측치 확인

#### 컬럼별 결측치 개수/비율
- 시군구, 법정동, 유형, 용도지역, 전용/연면적(㎡), 건축물주용도, 거래금액(만원), 계약연월 : 0개(0.0%)
- 층: 1,408개(46.01%)
<br>
<br>
※상업/업무용 부동산 데이터에서 '층' 정보가 NaN인 경우<br>
(1) '통건물' 매매인 경우(건물 유형이 일반이거나 면적이 수백 ㎡ 단위로 크다면 통건물 매매일 확률 높음)<br>
(2) '토지'만 거래된 경우(용도지역은 '상업지역'인데 건물 면적이 0이거나 거의 없다면 토지 거래)<br>
(3) 집합건물 중 '층' 구분이 모호한 경우(지상/지하 구분 누락 또는 행정 오류)

- 건물 유형이 일반인 부동산 거래 데이터 모두 층이 NaN으로 파악. '통건물' 매매로 분류.

- (질문) (2)와 (3)의 경우(건물 유형이 집합인 부동산 거래 중 층이 NaN인 데이터) 어떻게 처리하는 게 좋을까(해당 조건에 속하는 데이터들이 많고 거래량 추출이 주목적이므로 '지하'로 대체 가능?)
- (튜터님 답변) 현재 '층' 관련 정보는 거래금액의 이상치를 판단할 때 활용하는 보조지표 수준, 정확한 정보를 파악할 수 없는 행들은 NaN, '지하(현 데이터에서 이미 '지하'로 분류된 행이 없을 경우)', '구분 안 됨' 등으로 처리해도 무방.

- 결론: (일반) NaN → 'whole_building', (집합) NaN → 'unknown'으로 처리

In [69]:
trans_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   시군구        3060 non-null   str    
 1   법정동        3060 non-null   str    
 2   유형         3060 non-null   str    
 3   용도지역       3060 non-null   str    
 4   전용/연면적(㎡)  3060 non-null   float64
 5   건축물주용도     3060 non-null   str    
 6   거래금액(만원)   3060 non-null   int64  
 7   층          1652 non-null   float64
 8   계약연월       3060 non-null   str    
dtypes: float64(2), int64(1), str(6)
memory usage: 215.3 KB


In [70]:
trans_df['층'].isna().sum()

np.int64(1408)

In [71]:
trans_df['층'].isna().mean() * 100

np.float64(46.01307189542484)

In [72]:
floor_null = trans_df[(trans_df['유형'] == '집합') & (trans_df['층'].isna())] 
floor_null

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월
3,경기도 성남시 분당구,서현동,집합,근린상업,76,제1종근린생활,37000,NaN,2023-01
5,경기도 성남시 분당구,서현동,집합,근린상업,131,제1종근린생활,64247,NaN,2023-01
6,경기도 성남시 분당구,서현동,집합,준주거,23,제2종근린생활,23000,NaN,2023-01
7,경기도 성남시 분당구,서현동,집합,중심상업,35,업무,13700,NaN,2023-01
8,경기도 성남시 분당구,서현동,집합,중심상업,67,제2종근린생활,20500,NaN,2023-01
...,...,...,...,...,...,...,...,...,...
3022,경기도 성남시 분당구,수내동,집합,준주거,17,제1종근린생활,51000,NaN,2025-12
3034,경기도 성남시 분당구,정자동,집합,근린상업,22,제1종근린생활,4800,NaN,2025-12
3036,경기도 성남시 분당구,정자동,집합,제3종일반주거,41,제1종근린생활,53000,NaN,2025-12
3038,경기도 성남시 분당구,정자동,집합,중심상업,864,업무,645000,NaN,2025-12


In [73]:
trans_df[(trans_df['유형'] == '집합') & (trans_df['층'].notna())] 

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월
0,경기도 성남시 분당구,구미동,집합,근린상업,30,제2종근린생활,69500,1,2023-01
1,경기도 성남시 분당구,금곡동,집합,중심상업,45,제2종근린생활,69000,1,2023-01
2,경기도 성남시 분당구,서현동,집합,근린상업,20,교육연구,9753,2,2023-01
4,경기도 성남시 분당구,서현동,집합,근린상업,76,제1종근린생활,37000,2,2023-01
10,경기도 성남시 분당구,야탑동,집합,근린상업,147,제2종근린생활,52199,5,2023-01
...,...,...,...,...,...,...,...,...,...
3055,경기도 성남시 중원구,성남동,집합,일반상업,52,제2종근린생활,96000,1,2025-12
3056,경기도 성남시 중원구,성남동,집합,일반상업,79,제1종근린생활,23000,2,2025-12
3057,경기도 성남시 중원구,성남동,집합,제2종일반주거,101,기타,47000,1,2025-12
3058,경기도 성남시 중원구,여수동,집합,근린상업,25,업무,13600,10,2025-12


In [74]:
trans_df[(trans_df['유형'] == '일반') & (trans_df['층'].isna())] 

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월
24,경기도 성남시 수정구,태평동,일반,제2종일반주거,274,제2종근린생활,160000,NaN,2023-01
25,경기도 성남시 수정구,태평동,일반,제2종일반주거,1339,제1종근린생활,380000,NaN,2023-01
31,경기도 성남시 중원구,성남동,일반,일반상업,1178,숙박,615000,NaN,2023-01
180,경기도 성남시 분당구,운중동,일반,보전녹지,499,제1종근린생활,530000,NaN,2023-03
194,경기도 성남시 수정구,신흥동,일반,제3종일반주거,161,제2종근린생활,62000,NaN,2023-03
...,...,...,...,...,...,...,...,...,...
3029,경기도 성남시 분당구,운중동,일반,자연녹지,664,제2종근린생활,437500,NaN,2025-12
3033,경기도 성남시 분당구,이매동,일반,자연녹지,480,제2종근린생활,355970,NaN,2025-12
3040,경기도 성남시 수정구,수진동,일반,준주거,3634,제1종근린생활,1677520,NaN,2025-12
3047,경기도 성남시 중원구,금광동,일반,제2종일반주거,136,제1종근린생활,88326,NaN,2025-12


In [75]:
trans_df[(trans_df['유형'] == '일반') & (trans_df['층'].notna())] 

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월


In [76]:
trans_df['층'] = trans_df['층'].astype('object')
trans_df.loc[(trans_df['유형'] == '집합') & (trans_df['층'].isna()), '층'] = 'unknown'
trans_df.loc[(trans_df['유형'] == '일반') & (trans_df['층'].isna()), '층'] = 'whole_building'

In [77]:
trans_df.loc[(trans_df['유형'] == '집합') & (trans_df['층'] == 'unknown')]

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월
3,경기도 성남시 분당구,서현동,집합,근린상업,76,제1종근린생활,37000,unknown,2023-01
5,경기도 성남시 분당구,서현동,집합,근린상업,131,제1종근린생활,64247,unknown,2023-01
6,경기도 성남시 분당구,서현동,집합,준주거,23,제2종근린생활,23000,unknown,2023-01
7,경기도 성남시 분당구,서현동,집합,중심상업,35,업무,13700,unknown,2023-01
8,경기도 성남시 분당구,서현동,집합,중심상업,67,제2종근린생활,20500,unknown,2023-01
...,...,...,...,...,...,...,...,...,...
3022,경기도 성남시 분당구,수내동,집합,준주거,17,제1종근린생활,51000,unknown,2025-12
3034,경기도 성남시 분당구,정자동,집합,근린상업,22,제1종근린생활,4800,unknown,2025-12
3036,경기도 성남시 분당구,정자동,집합,제3종일반주거,41,제1종근린생활,53000,unknown,2025-12
3038,경기도 성남시 분당구,정자동,집합,중심상업,864,업무,645000,unknown,2025-12


In [78]:
trans_df.loc[(trans_df['유형'] == '일반') & (trans_df['층'] == 'whole_building')]

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월
24,경기도 성남시 수정구,태평동,일반,제2종일반주거,274,제2종근린생활,160000,whole_building,2023-01
25,경기도 성남시 수정구,태평동,일반,제2종일반주거,1339,제1종근린생활,380000,whole_building,2023-01
31,경기도 성남시 중원구,성남동,일반,일반상업,1178,숙박,615000,whole_building,2023-01
180,경기도 성남시 분당구,운중동,일반,보전녹지,499,제1종근린생활,530000,whole_building,2023-03
194,경기도 성남시 수정구,신흥동,일반,제3종일반주거,161,제2종근린생활,62000,whole_building,2023-03
...,...,...,...,...,...,...,...,...,...
3029,경기도 성남시 분당구,운중동,일반,자연녹지,664,제2종근린생활,437500,whole_building,2025-12
3033,경기도 성남시 분당구,이매동,일반,자연녹지,480,제2종근린생활,355970,whole_building,2025-12
3040,경기도 성남시 수정구,수진동,일반,준주거,3634,제1종근린생활,1677520,whole_building,2025-12
3047,경기도 성남시 중원구,금광동,일반,제2종일반주거,136,제1종근린생활,88326,whole_building,2025-12


In [83]:
trans_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   시군구        3060 non-null   str    
 1   법정동        3060 non-null   str    
 2   유형         3060 non-null   str    
 3   용도지역       3060 non-null   str    
 4   전용/연면적(㎡)  3060 non-null   float64
 5   건축물주용도     3060 non-null   str    
 6   거래금액(만원)   3060 non-null   int64  
 7   층          3060 non-null   object 
 8   계약연월       3060 non-null   str    
dtypes: float64(1), int64(1), object(1), str(6)
memory usage: 215.3+ KB


## 이상치 확인

#### 1. 거래금액

- 최솟값: 200만원(2,000,000원), 최댓값: 1조9820억4140만원(1,982,041,400,000원)
- 상업용 부동산 매매금액이 200만원은 너무 작음. 보수적으로 접근한다 해도 최소 1,000만원 ~ 1,500만원부터 시작

In [79]:
trans_df['거래금액(만원)'].describe()

count        3060
mean       236973
std       4024818
min           200
25%         19000
50%         38000
75%         76000
max     198204140
Name: 거래금액(만원), dtype: float64

In [80]:
trans_df.sort_values(by='거래금액(만원)').head()

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월
397,경기도 성남시 중원구,성남동,집합,일반상업,4,판매,200,4,2023-05
891,경기도 성남시 중원구,성남동,집합,일반상업,5,판매,500,unknown,2023-11
1559,경기도 성남시 분당구,서현동,집합,근린상업,9,제1종근린생활,1000,unknown,2024-07
2495,경기도 성남시 중원구,성남동,집합,일반상업,5,판매,1000,2,2025-06
815,경기도 성남시 분당구,구미동,집합,근린상업,13,제1종근린생활,1200,2,2023-11


In [81]:
# 상·하위 10% 중에서도 극단인 값
lower_10 = np.percentile(trans_df['거래금액(만원)'], 10)
upper_90 = np.percentile(trans_df['거래금액(만원)'], 90)

subset = trans_df[(trans_df['거래금액(만원)'] <= lower_10) | (trans_df['거래금액(만원)'] >= upper_90)]

q1 = subset['거래금액(만원)'].quantile(0.25)
q3 = subset['거래금액(만원)'].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

extreme = subset[(subset['거래금액(만원)'] < lower) | (subset['거래금액(만원)'] > upper)]
extreme

,시군구,법정동,유형,용도지역,전용/연면적(㎡),건축물주용도,거래금액(만원),층,계약연월
58,경기도 성남시 분당구,삼평동,집합,준주거,940,업무,1245000,10,2023-02
223,경기도 성남시 분당구,백현동,일반,중심상업,87902,업무,69135000,whole_building,2023-04
268,경기도 성남시 분당구,운중동,일반,준주거,865,제2종근린생활,1050000,whole_building,2023-04
330,경기도 성남시 분당구,삼평동,집합,준주거,1047,교육연구,1150000,unknown,2023-05
355,경기도 성남시 분당구,운중동,일반,준주거,3332,제2종근린생활,2150000,whole_building,2023-05
403,경기도 성남시 분당구,구미동,집합,근린상업,1645,업무,1307037,unknown,2023-06
404,경기도 성남시 분당구,구미동,집합,근린상업,1904,업무,1777610,unknown,2023-06
405,경기도 성남시 분당구,구미동,집합,근린상업,2447,업무,2117600,unknown,2023-06
406,경기도 성남시 분당구,구미동,집합,근린상업,2764,업무,2320025,unknown,2023-06
422,경기도 성남시 분당구,삼평동,집합,준주거,1047,교육연구,1251900,6,2023-06


#### 2. 전용/연면적(㎡)

- 최솟값: 4㎡(1.21평), 최댓값: 197,237㎡(약 59,664평)
- 상업용 부동산 거래 면적 4㎡은 너무 작음. 전용/연면적(㎡)이 작을수록 거래금액도 비정상적으로 낮음(거래금액-면적: 상관관계 보면서 극단값 보기)
- (질문 사항) 얼마부터 이상치로 볼 것인가 판단 기준 필요

In [82]:
trans_df['전용/연면적(㎡)'].describe()

count     3060
mean       311
std       4382
min          4
25%         26
50%         50
75%        101
max     197237
Name: 전용/연면적(㎡), dtype: float64

## 파생변수 생성: 거래량

결측치/이상치 처리 후 법정동 단위로 집계 예정